# DSCI 525 - Web and Cloud Computing Next Steps Helper

## Helper Video from Gittu on General AWS Walkthrough (from Milestone 2)
[Here](https://www.youtube.com/watch?v=1koYniWo53g) is the Week 2 helper video, with timestamps for the main setup steps:

### For us, here are the relevent sections
- [0:00](https://www.youtube.com/watch?v=1koYniWo53g&t=0s) 1-Complete the lease/sandbox setup
- [0:18](https://www.youtube.com/watch?v=1koYniWo53g&t=18s) 2-Create your S3 bucket
- [1:13](https://www.youtube.com/watch?v=1koYniWo53g&t=73s) 3-Setup your EC2 instance
- [2:40](https://www.youtube.com/watch?v=1koYniWo53g&t=160s) 4-Setup your JupyterHub
- [4:27](https://www.youtube.com/watch?v=1koYniWo53g&t=267s) 5-Set up the server/logging in
- [8:25](https://www.youtube.com/watch?v=1koYniWo53g&t=505s) 6-Setup AWS CLI


**Keep in mind:**

- Use region `ca-central-1` for the services in all milestones.
- Use only the default VPC and default subnet.
- Use a single running instance whenever possible to control cost.
- If you terminate your EC2 instance, data stored only on that instance or EBS volume will be lost. Save important data to S3 and download your notebooks when needed.

# Create an S3 bucket

> Here is the link to specific timestamp in the helper video: [0:18](https://www.youtube.com/watch?v=1koYniWo53g&t=18s).

Create the S3 bucket first so that you already have a destination ready when it is time to move data.

- Name your bucket `mds-s3-<name>`. For example: `mds-s3-gittu`.
- Uncheck **Block all public access** and acknowledge the warning.
- Create a folder inside the bucket called `output`.
- Under the **Permissions** tab, in **Bucket policy**, use the policy below. Update the bucket name to match your own bucket.

```json
{
  "Version": "2012-10-17",
  "Id": "Policy1649284381437",
  "Statement": [
    {
      "Sid": "Stmt1649284379371",
      "Effect": "Allow",
      "Principal": {
        "AWS": "*"
      },
      "Action": "s3:*",
      "Resource": [
        "arn:aws:s3:::mds-s3-gittu",
        "arn:aws:s3:::mds-s3-gittu/*"
      ]
    }
  ]
}
```


# Setup an EC2 instance

> Here is the link to specific timestamp in the helper video: [1:13](https://www.youtube.com/watch?v=1koYniWo53g&t=73s).

Now set up the cloud machine that your team will use.

- Region: `ca-central-1`
- Name of the instance: `mds-yourname` (for example `mds-gittu`)
- AMI: `Ubuntu Server 22.04 LTS (HVM)`
- Instance type: `t3a.xlarge`
- Architecture: `64-bit (x86)`
- Create a `.pem` key pair and download the private key file to your computer. You will need it to connect to your instance.
- Allow access from SSH, HTTPS, and HTTP.
- Storage: `30 GB`
- Storage type: `General Purpose SSD (gp3)`
- Install TLJH in your instance by adding the setup instructions below to **User Data**.

```bash
#!/bin/bash
curl -L https://tljh.jupyter.org/bootstrap.py \
  | sudo python3 - \
    --admin gittu
```



# Setup JupyterHub

> Here is the link to specific timestamp in the helper video: [2:40](https://www.youtube.com/watch?v=1koYniWo53g&t=160s).

- Open JupyterHub by copying the **Public IPv4 address** of your EC2 instance into your browser. Make sure you use `http`.
- Log in using the admin username that you gave in your EC2 **User Data** and the password `ubuntumds`.
- Add the partners you want to collaborate with so they can work in the environment. You can do this from **Files -> Hub Control Panel -> Admin -> Add Users**. Check **Admin** as well.

Install the packages needed:

- go to **Home** and then click **MyServer**):
- Open a terminal from the JupyterHub interface so that the packages are installed in the correct environment. Use `-E` so the installed packages are available to all JupyterHub users.

```bash
sudo -E pip install pandas
sudo -E pip install pyarrow
sudo -E pip install s3fs
```

Check the TLJH user environment guide [here](https://tljh.jupyter.org/en/latest/howto/user-env/user-environment.html)(if you want more details).
    

# Setup the server

> Here is the link to specific timestamp in the helper video: [4:27](https://www.youtube.com/watch?v=1koYniWo53g&t=267s).


At this point you have a machine running in the cloud. Next, make it usable for anyone you choose.

Log in to your EC2 instance from your laptop terminal using your private key and the public hostname.

Use the AWS guide below for creating additional Linux user accounts on the instance:
Check [this for more details](https://repost.aws/knowledge-center/new-user-accounts-linux-instance)

You can do below for that:

```bash
sudo adduser maria --disabled-password
sudo mkdir -p /home/maria/.ssh
sudo vi /home/maria/.ssh/authorized_keys
# Now add the public key of maria to the authorized_keys file and save it.
# maria can do this by doing ssh-keygen -y -f maria.pem
sudo chown -R maria:maria /home/maria/.ssh
sudo chmod 700 /home/maria/.ssh
sudo chmod 600 /home/maria/.ssh/authorized_keys
```

After setup, each partner should be able to log in using their own username and private key, for example:

```bash
ssh -i <path_to_private_key> maria@<your-ec2-hostname>
```

If you need a refresher on key pairs, refer back to DSCI 521.

5.2) Set up a common shared space. This is also done by the root user managing the server setup.

```bash
sudo mkdir -p /srv/data/my_shared_data_folder
sudo chmod 777 /srv/data/my_shared_data_folder
```

For more detail, [see the TLJH shared-data guide](https://tljh.jupyter.org/en/latest/howto/content/share-data.html)



# Setup AWS CLI

> Here is the link to specific timestamp in the helper video: [8:25](https://www.youtube.com/watch?v=1koYniWo53g&t=505s).

Now configure AWS CLI so that you can interact with AWS services from your cloud machine whenever needed.

- Go back to the student access portal (eg `Student_Sandbox_019`) and click **Access keys**.
- (only need to check this for additional reading) Use the AWS IAM Identity Center route described [here](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-sso-tutorial.html).
  
- Use the SSO start URL from the access-keys page and the region `ca-central-1`.

First, install AWS CLI on the EC2 instance. Make sure you use the install instructions for the correct architecture [from the official documentation](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html).

Here is an example of the install commands for the `x86_64` architecture, ubuntu machines.

```bash
sudo apt install unzip
curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
unzip awscliv2.zip
sudo ./aws/install
aws --version
```

Then configure AWS CLI using the SSO method. [Here](https://docs.aws.amazon.com/cli/latest/userguide/cli-configure-sso-tutorial.html) is the official document if you want to read more.

Start with:

```bash
aws configure sso --use-device-code
```

Here is an example of the prompts and responses:

```text
SSO session name (Recommended): mds-test
SSO start URL [None]: https://ubclthubcourses.awsapps.com/start/#
SSO region [None]: ca-central-1
SSO registration scopes [sso:account:access]: sso:account:access
```

Continue through the browser login flow when prompted and paste the access code from the terminal.

```text
Default client Region [None]: ca-central-1
CLI default output format (json if not specified) [None]:
Profile name [lticisb_IsbUsersPS-441243247526]: gittu_profile
```

To test the profile, run:

```bash
aws sts get-caller-identity --profile gittu_profile
aws s3 ls --profile gittu_profile
```

Make sure the CLI works before you move on to the data steps. Listing your bucket successfully is enough to confirm the setup.

### (Optional) Share AWS CLI SSO Config with Partner

If your partner will use your EC2 server to interact with AWS, you can share your AWS CLI configuration with them.

```bash
sudo mkdir -p /home/maria/.aws
sudo cp /home/ubuntu/.aws/config /home/maria/.aws/config
sudo chown -R maria:maria /home/maria/.aws
sudo chmod 700 /home/maria/.aws
sudo chmod 600 /home/maria/.aws/config
```

(Optional) Partner Login (when needed)

When your partner wants to use AWS CLI from the server, they should run:

```
aws sso login --profile gittu_profile --use-device-code
```

# Move  data to S3

> Here is the link to specific timestamp in the helper video: [12:39](https://www.youtube.com/watch?v=1koYniWo53g&t=759s).

8.1) Use the bucket you created earlier: `mds-s3-<name>`.

8.2) Make sure the `output` folder exists.

8.3) Upload `observed_daily_rainfall_SYD.csv` from your Milestone 2 data folder to your S3 bucket.

8.4) Upload the parquet file you downloaded in Step 7 (`combined_model_data_parti.parquet`) to S3 using AWS CLI.

Use the CLI approach for the parquet upload. For example:

```bash
aws s3 cp combined_model_data_parti.parquet/ s3://mds-s3-gittu/combined_model_data_parti.parquet/ --recursive --profile gittu_profile
```

You should practice both ways of working with S3:
- upload `observed_daily_rainfall_SYD.csv` directly from the AWS web console. You can find it in the folder from Milestone 1.

You will use the `output/` folder in the next section when saving the machine learning-ready file.

